# Convert Excel Files to CSV

This notebook converts .xlsx files in landing folders to .csv format for ingestion.

In [ ]:
# Parameters
period = "2026-03"
dfm_id = "wh_ireland"  # Change this for each DFM

print(f"[INFO] Converting Excel files for {dfm_id}, period={period}")

# Imports
import pandas as pd
from pyspark.sql import functions as F

landing_path = f"Files/landing/period={period}/dfm={dfm_id}/source/"
print(f"[INFO] Landing path: {landing_path}")

In [ ]:
# Discover Excel files
print("[STEP 1] Discovering Excel files...")

try:
    entries = [f for f in mssparkutils.fs.ls(landing_path) if not f.isDir]
    xlsx_files = [f for f in entries if f.name.lower().endswith((".xlsx", ".xls"))]
    print(f"[INFO] Found {len(xlsx_files)} Excel file(s)")
    for f in xlsx_files:
        print(f"  - {f.name}")
except Exception as e:
    print(f"[ERROR] Could not access landing path: {str(e)}")
    raise

if len(xlsx_files) == 0:
    print("[ERROR] No Excel files found")
    raise Exception("NO_XLSX_FILES")

In [ ]:
# Convert each Excel file to CSV
print("\n[STEP 2] Converting to CSV...")

for xlsx_file in xlsx_files:
    try:
        print(f"\n[INFO] Processing: {xlsx_file.name}")
        
        # Read Excel file with pandas
        # Note: First row is groupings (skip), second row is headers
        pdf = pd.read_excel(
            xlsx_file.path,
            skiprows=1,  # Skip row 1 (groupings)
            dtype=str,   # Keep as strings
            keep_default_na=False
        )
        
        print(f"  - Loaded {len(pdf)} rows, {len(pdf.columns)} columns")
        
        # Generate CSV filename
        csv_name = xlsx_file.name.lower().replace(".xlsx", ".csv").replace(".xls", ".csv")
        
        # For WH Ireland, map to expected filename
        if "standard life" in csv_name or "valuation" in csv_name:
            csv_name = "Notification.csv"  # Main positions file
        
        csv_path = f"{landing_path}{csv_name}"
        
        # Convert to Spark DataFrame and write as CSV
        sdf = spark.createDataFrame(pdf)
        
        # Write to CSV (will create a folder with CSV inside)
        temp_output = f"Files/temp_csv_{dfm_id}"
        sdf.coalesce(1).write.mode("overwrite").option("header", "true").csv(temp_output)
        
        # Find the generated CSV file
        csv_files = [f for f in mssparkutils.fs.ls(temp_output) if f.name.endswith(".csv")]
        if csv_files:
            # Copy to landing folder with correct name
            mssparkutils.fs.cp(csv_files[0].path, csv_path)
            print(f"  ✓ Created: {csv_name}")
        
        # Clean up temp folder
        mssparkutils.fs.rm(temp_output, recurse=True)
        
    except Exception as e:
        print(f"  ✗ Failed: {str(e)}")

print("\n[SUCCESS] Conversion complete")

In [ ]:
# Verify CSV files created
print("\n[STEP 3] Verifying CSV files...")

try:
    entries = [f for f in mssparkutils.fs.ls(landing_path) if not f.isDir]
    csv_files = [f for f in entries if f.name.lower().endswith(".csv")]
    print(f"[INFO] Found {len(csv_files)} CSV file(s)")
    for f in csv_files:
        print(f"  ✓ {f.name}")
except Exception as e:
    print(f"[ERROR] {str(e)}")

print("\n[INFO] Ready for ingestion!")